# Node-held-out modularity — testing scProto on cells it has never seen at all

**Reviewer concern this answers** (F5RB, follow-up round — full thread in
`../comments1/reviewer1_F5RB_round2_reply.md` and `../my-notes/comments1/rev1.txt`):

> "held-out edges from the same graph are not fully independent" — F5RB, responding to
> the edge-masking design in `heldout_edge_modularity.ipynb`.

Our round-2 reply told F5RB we were "exploring whether an independent-graph version of
this test is feasible." This notebook is that experiment.

**What was independent before, and what wasn't.** `heldout_edge_modularity.ipynb` hides
20% of the affinity graph's *edges* from training, but every *cell* — including both
endpoints of every hidden edge — is still present in the graph that scProto trains on,
still gets a UMAP-edge gradient from its *visible* edges, and still shapes the encoder
through Stage 1 pretraining and Stage 2's community loss. A held-out edge is independent
of the training *signal for that specific pair*, but not of the training *process* as a
whole — both cells it connects were still seen.

**What's different here.** A fraction of *cells* — not edges — are set aside as test
nodes before anything else happens. The affinity graph scProto trains on is built from
train cells only, so a test cell is never a node in that graph, never enters the UMAP
edge sampler, and contributes no gradient anywhere, at any stage. scProto's encoder is
graph-free at inference (`trainer.py:257` `encode_adata` — expression and batch/condition
one-hots only, no adjacency matrix in its forward path), so once training finishes we run
the *frozen* encoder forward on the held-out cells' expression alone and ask: does the
resulting embedding still place them correctly relative to a community structure the
model was never trained or early-stopped against? This is model generalization to unseen
cells, not just unseen edges between seen cells — the strongest independence this
architecture can support without a second, held-out dataset entirely.

**Rows in the comparison:**

1. **scProto (node-holdout)** — trained end-to-end (Stage 1 + Stage 2) on train cells
   only; test cells scored via a frozen-encoder forward pass, never touched during
   training.
2. **Leiden (scPoli Stage-1)** / **Leiden (scVI, Gaussian)** — the "zero affinity
   supervision" controls from `heldout_edge_modularity.ipynb`, reused as-is: both
   encoders never consume the affinity graph in any form, so their existing E1 Leiden
   assignments (fit on the *full* dataset) are simply re-scored against this notebook's
   test-node edges. **Caveat, stated up front rather than glossed over**: unlike scProto
   (node-holdout), these two rows' own encoders *did* see every test cell's raw
   expression during their own (graph-free) training — they just never received any
   graph signal. That is a real asymmetry in scProto's favor being tested here; a win
   against these rows shows affinity supervision transfers to unseen structure, not that
   scProto's encoder alone is better at generalizing to unseen expression.
3. **No SEACells row.** This is not an oversight — it's the point worth stating to the
   reviewer directly. SEACells has no encoder; its kernel and archetypal fit are defined
   only over the cells present when the kernel is built. There is no way to place a cell
   it never saw, so it is *structurally* excluded from this test, not just weaker on it.
   That asymmetry (inductive vs. transductive) is itself evidence for the paper's
   framing of scProto as a metacell method with an encoder, not a clustering-on-a-fixed-
   graph method.

**Decisions, matching the discipline `heldout_edge_modularity.ipynb` already
established:**
- Node holdout is uniform/random within each batch (stratified only by `batch_key`, not
  by cell type) — deliberately not targeted at rare types, for the same reason as the
  edge-masking notebook: hiding *more* rare-cell nodes would invite a "you engineered the
  test set" objection. Rare-vs-common stratification happens only *after* the random
  split, as a post-hoc diagnostic.
- A per-batch floor (`MIN_TRAIN_PER_BATCH`) keeps enough train cells in every batch so
  Stage-1 scPoli conditioning still has signal per batch, and so the frozen-encoder
  inference path on test cells never silently falls back to scPoli's query-mapping
  retrain logic (`Trainer.check_conditions_compatible` — see the `encode_and_assign_new_cells`
  helper below; we assert this explicitly rather than trust it silently).
- Early stopping during the node-holdout scProto run uses modularity on the *train-only*
  graph exclusively — test cells are structurally absent from that graph (not just
  masked), so this is automatic rather than a masking rule to enforce.
- **Evaluation graph is the untouched, already-existing full-dataset affinity graph**
  (`t_base.train_ds.aff_raw`, the same one the paper's own headline modularity number
  uses) — no new graph is built for scoring. Test-node edges are simply the subset of
  that graph touching at least one held-out cell.
- Same Newman weighted modularity formula, same per-batch mean/std convention, as the
  paper's headline number and `heldout_edge_modularity.ipynb`.

**Scope: all 3 RNA-seq datasets** (pancreas, lung, pbmc-immune) — matching
`batch_correct_then_cluster_baselines.ipynb` (E1) and `heldout_edge_modularity.ipynb`, so
the same saved Leiden assignments can be reused without retraining.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q scarches faiss-gpu-cu12 scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 150.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 126.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 138.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 159.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 106.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 711.2/711.2 kB 57.6 MB/s eta 0:00:00
   ━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 145.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seacells 0.3.3 requires pyranges, which is not installed.
anndata 0.13.2 requires scipy!=1.17.0,>=1.14, but you have scipy 1.13.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.66.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 whi

In [13]:
# IMPORTANT: restart the runtime after this cell before running the cells below --
# numpy/scipy/anndata are C-extension linked, an in-process upgrade alone won't
# reliably take effect on already-imported modules.

In [14]:
# Ground truth for "did the install cell above actually work" -- pip's own log is
# noisy (resolver backtracking prints "Getting requirements to build wheel" errors
# for discarded candidate versions even on a fully successful install), so eyeballing
# it is unreliable. Actually importing every package we just installed is the real
# test.
_checks = {
    'numpy': 'numpy', 'scipy': 'scipy', 'anndata': 'anndata', 'scanpy': 'scanpy',
    'scarches': 'scarches', 'scvi-tools': 'scvi', 'seacells': 'SEACells',
    'palantir': 'palantir', 'scib-metrics': 'scib_metrics', 'leidenalg': 'leidenalg',
    'python-igraph': 'igraph', 'umap-learn': 'umap', 'harmonypy': 'harmonypy',
    'faiss-cpu': 'faiss',
}
_failed = []
for pkg_name, import_name in _checks.items():
    try:
        mod = __import__(import_name)
        ver = getattr(mod, '__version__', '?')
        print(f"  OK   {pkg_name:16s} (import {import_name}, version {ver})")
    except Exception as e:
        _failed.append(pkg_name)
        print(f"  FAIL {pkg_name:16s} (import {import_name}): {type(e).__name__}: {e}")

if _failed:
    print(f"\n{len(_failed)} package(s) failed to import: {_failed} -- re-run that "
          f"package's specific pip install line above and check its full error "
          f"output before proceeding.")
else:
    print(f"\nAll {len(_checks)} packages import cleanly -- safe to continue.")

  OK   numpy            (import numpy, version 2.2.6)
  OK   scipy            (import scipy, version 1.13.1)
  OK   anndata          (import anndata, version 0.13.2)
  OK   scanpy           (import scanpy, version 1.12.3)
  OK   scarches         (import scarches, version 0.6.1)
  OK   scvi-tools       (import scvi, version 1.5.0.post1)
  OK   seacells         (import SEACells, version 0.3.3)
  OK   palantir         (import palantir, version 1.4.5)
  OK   scib-metrics     (import scib_metrics, version 0.6.0)
  OK   leidenalg        (import leidenalg, version 0.12.0)
  OK   python-igraph    (import igraph, version 1.0.0)
  OK   umap-learn       (import umap, version 0.5.12)
  OK   harmonypy        (import harmonypy, version 2.0.0)
  OK   faiss-cpu        (import faiss, version 1.14.1)

All 14 packages import cleanly -- safe to continue.


In [15]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


In [16]:
# Extra imports not already covered by nb_setup.py (which already pulls in
# run_mc_task, show_table, clean_run_names, etc. via
# `from interpretable_ssl.evaluation.paper_figures import *`).
import os
import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import anndata
import scanpy as sc

from interpretable_ssl.experiments.tasks import run_mc_task, LAMBDA_PROTO_UMAP_PRECON
from interpretable_ssl.evaluation.mc_metric_utils import get_rare, calc_modularity_per_batch
from interpretable_ssl.configs.paths import get_dataset_model_dir
from interpretable_ssl.datasets.dataset_configs import DATASETS

print("extra imports ready")

extra imports ready


## Config

In [17]:
RNA_SEQ_DATASETS = ['pancreas', 'lung', 'pbmc-immune']
AFFINITY = 'arbf'

FRAC_TEST = 0.20              # fraction of CELLS (nodes) held out entirely, not edges
MIN_TRAIN_PER_BATCH = 50      # floor on train cells retained per batch, so Stage-1 scPoli
                               # conditioning still has signal for every batch and the
                               # frozen-encoder inference path never needs to fall back to
                               # scPoli's query-mapping retrain logic (see
                               # encode_and_assign_new_cells below)
SEED = 0
RARE_QUANTILE = 0.25          # matches mc_metric_utils.get_rare's convention (get_rare
                               # currently ignores its own thr= param and always
                               # recomputes internally at 0.25 -- kept here for
                               # readability only, not actually wired through)

TAG = f'{AFFINITY}_nodeheldout{int(FRAC_TEST * 100)}_seed{SEED}'

NODE_HELDOUT_DATA_DIR = os.path.join(os.environ['CODE_DIR'], 'node_heldout_data')
os.makedirs(NODE_HELDOUT_DATA_DIR, exist_ok=True)

CVAE_EPOCHS = 50
BATCH_SIZE = 1024

def common_kwargs():
    return dict(
        cvae_epochs=CVAE_EPOCHS,
        train_epochs=50,
        eval_freq=3,
        patience=6,
        batch_size=BATCH_SIZE,
        umap_steps_per_epoch=500,
        lambda_config=LAMBDA_PROTO_UMAP_PRECON | {'nassoc_agg': 'max'},
    )

# The node-holdout scProto run -- the new thing this notebook produces per dataset.
# False trains fresh; flip to True on a re-run (after a first successful run) to just
# reload the checkpoint instead of retraining.
LOAD_NODEHELDOUT = True

# scPoli-Stage1 / scVI-Gaussian two-step baselines reused from
# batch_correct_then_cluster_baselines.ipynb (E1) -- Leiden assignments only. See
# load_two_step_leiden_assignments' docstring below for why SEACells is structurally
# excluded from this notebook rather than just omitted. Add 'harmony' / 'bbknn' /
# 'combat' to this list the same way (identical E1 folder-tag convention) if a reviewer
# specifically asks for those rows too.
TWO_STEP_METHODS = ['stage1z', 'scvi']
METHOD_DISPLAY_NAMES = {
    'stage1z': 'scPoli (Stage-1)',
    'scvi': 'scVI (Gaussian)',
}

TAG

'arbf_nodeheldout20_seed0'

## Helper functions

In [18]:
def split_nodes_train_test(ad, batch_key, frac_test=0.2, min_train_per_batch=50, seed=0):
    '''Split cell indices into train/test nodes, stratified by batch only.

    Not an edge split -- test cells are removed as NODES, so they cannot appear in any
    affinity graph built on the train-only AnnData afterwards. Stratified by batch (not
    by cell type) so every batch keeps at least `min_train_per_batch` train cells --
    keeps Stage-1 scPoli conditioning meaningful per batch and keeps every test cell's
    batch/condition value a subset of what the trained model has seen (required for
    encode_and_assign_new_cells' frozen-forward-pass assumption to hold). Uniform random
    within each batch, not targeted at rare cell types -- same non-cherry-picking
    principle as heldout_edge_modularity.ipynb's split_affinity_edges.

    Returns:
        (train_idx, test_idx): int arrays of positional indices into `ad`.
    '''
    rng = np.random.default_rng(seed)
    n = ad.n_obs
    batches = ad.obs[batch_key].values
    test_mask = np.zeros(n, dtype=bool)

    for b in pd.unique(batches):
        idx_b = np.where(batches == b)[0]
        n_b = len(idx_b)
        n_test_b = int(round(frac_test * n_b))
        n_test_b = min(n_test_b, max(0, n_b - min_train_per_batch))
        if n_test_b < round(frac_test * n_b):
            print(f"  batch {b!r}: capped test count to {n_test_b}/{n_b} to keep "
                  f">= {min_train_per_batch} train cells")
        chosen = rng.choice(idx_b, size=n_test_b, replace=False)
        test_mask[chosen] = True

    test_idx = np.where(test_mask)[0]
    train_idx = np.where(~test_mask)[0]
    print(f"node split: {len(test_idx)}/{n} cells held out as test nodes "
          f"({test_mask.mean():.1%}, target was {frac_test:.0%}), "
          f"{len(train_idx)} train cells remain")
    return train_idx, test_idx

In [19]:
def modularity_on_edges(A, assignments):
    '''Weighted Newman modularity of `assignments` scored against adjacency `A`.

    Q = (1/2m) * sum_k [ e_k - d_k^2 / (2m) ]
    Identical formula to Trainer.modularity() / mc_metric_utils.compute_modularity() /
    heldout_edge_modularity.ipynb's copy of this function -- duplicated here (second
    consumer) rather than imported, matching that notebook's own note that this hasn't
    been promoted into graph_generator.py yet.
    '''
    A = sp.csr_matrix(A)
    A = (A + A.T) / 2
    degrees = np.array(A.sum(axis=1)).ravel()
    two_m = degrees.sum()
    if two_m == 0:
        return 0.0
    Q = 0.0
    for c in np.unique(assignments):
        mask = (assignments == c)
        e_k = A[mask][:, mask].sum()
        d_k = degrees[mask].sum()
        Q += (e_k - d_k * d_k / two_m) / two_m
    return float(Q)


def held_out_same_cluster_rate(aff, assignments, cell_mask=None, require='any'):
    '''Fraction of edges in `aff` with both endpoints in the same cluster/prototype.

    Duplicated from heldout_edge_modularity.ipynb -- there it scores held-out *edges*;
    here it's called with the test-node-touching subgraph instead, same implementation.
    Complements modularity_on_edges() with a metric that stays interpretable on a small,
    sparse edge subset (e.g. rare-cell-touching edges only), where full Newman
    modularity's null-model term (global degree) gets noisy.

    Args:
        aff:          scipy sparse adjacency (e.g. the test-node-touching subgraph, or a
                      further-restricted subset of it).
        assignments:  (N,) array, cluster/prototype id per cell, aligned to aff's rows.
        cell_mask:    optional (N,) boolean array (e.g. a rare-cell-type mask).
        require:      'any'  -- keep edges where at least one endpoint is masked-True
                                 (use with a rare-type mask: "touches a rare cell").
                      'all'  -- keep edges where both endpoints are masked-True
                                 (use with a common-type mask: "both cells common").

    Returns: dict with 'rate', 'weighted_rate', 'n_edges'.
    '''
    A = sp.csr_matrix(aff)
    A = (A + A.T) / 2
    A = A.tocoo()
    upper = A.row < A.col
    rows, cols, vals = A.row[upper], A.col[upper], A.data[upper]

    if cell_mask is not None:
        cell_mask = np.asarray(cell_mask)
        if require == 'any':
            edge_mask = cell_mask[rows] | cell_mask[cols]
        elif require == 'all':
            edge_mask = cell_mask[rows] & cell_mask[cols]
        else:
            raise ValueError(f"require must be 'any' or 'all', got {require!r}")
        rows, cols, vals = rows[edge_mask], cols[edge_mask], vals[edge_mask]

    if len(rows) == 0:
        return {'rate': None, 'weighted_rate': None, 'n_edges': 0}

    same = assignments[rows] == assignments[cols]
    weighted_rate = float((same * vals).sum() / vals.sum()) if vals.sum() > 0 else None
    return {'rate': float(same.mean()), 'weighted_rate': weighted_rate, 'n_edges': int(len(rows))}


def subgraph_touching(A, mask):
    '''Restrict adjacency `A` to edges where at least one endpoint is in `mask`.

    Kept as a full (N, N) sparse matrix, not renumbered -- edges with neither endpoint
    in `mask` become exactly-zero entries rather than being dropped from the index
    space. This lets modularity_on_edges / calc_modularity_per_batch / the assignments
    array all stay aligned to the full N-cell ordering unmodified, the same convention
    heldout_edge_modularity.ipynb's aff_test already uses.
    '''
    A = sp.csr_matrix(A)
    A = (A + A.T) / 2
    A = A.tocoo()
    keep = mask[A.row] | mask[A.col]
    return sp.csr_matrix((A.data[keep], (A.row[keep], A.col[keep])), shape=A.shape)

In [20]:
def encode_and_assign_new_cells(t, ad_new):
    '''Assign never-before-seen cells to prototypes via the frozen trained encoder.

    Mirrors Trainer._get_assignments()'s 'proto' branch (scproto.py:2503-2534), but
    parameterized on an arbitrary AnnData instead of t.train_ds.adata -- the
    node-holdout counterpart of that method. ad_new's cells were not part of the
    dataset used to build the affinity graph or train the model at all, so
    encode_adata's default retrain_epochs=0 must resolve to a pure frozen forward
    pass (no gradient, no graph access) rather than silently invoking scPoli's
    query-mapping retrain path (trainer.py's adapt_model / load_query_data). That
    retrain path only triggers when adata's batch/condition values are NOT already a
    subset of what the model saw during training -- true here as long as the node
    split kept every batch present on the train side (split_nodes_train_test's
    min_train_per_batch guards exactly this) -- so we assert it explicitly rather
    than trust it silently.
    '''
    assert t.check_conditions_compatible(t.model, ad_new), (
        "ad_new has batch/condition values the trained model never saw during "
        "training -- encode_adata would silently retrain via scPoli's "
        "load_query_data instead of doing a frozen forward pass, which would leak "
        "graph-adjacent training signal into the 'unseen cell' evaluation. Check "
        "that split_nodes_train_test's min_train_per_batch kept every batch "
        "represented on the train side."
    )
    with torch.no_grad():
        z = t.encode_adata(ad_new, t.model, z_idx=1)
        scores = t.model.prototypes(z)
        assignments = scores.argmax(dim=1).cpu().numpy()
    return assignments

In [21]:
def _find_existing_leiden_dir(ds_id, tag):
    '''leiden's save folder name includes n_clusters (unknown ahead of time) --
    find it by prefix match, same convention heldout_edge_modularity.ipynb and
    batch_correct_then_cluster_baselines.ipynb use.
    '''
    base_dir = get_dataset_model_dir(ds_id)
    if not os.path.isdir(base_dir):
        return None
    for entry in sorted(os.listdir(base_dir)):
        if entry.startswith(f'leiden_{tag}_K') and os.path.exists(os.path.join(base_dir, entry, 'metrics.json')):
            return os.path.join(base_dir, entry)
    return None


def load_two_step_leiden_assignments(ds_id, method, cell_index):
    '''Load `method`'s already-saved Leiden cluster assignments from E1
    (batch_correct_then_cluster_baselines.ipynb), reindexed to `cell_index` (this
    notebook's own ad_full.obs_names order).

    Leiden-only, unlike heldout_edge_modularity.ipynb's load_two_step_assignments:
    SEACells has no encoder, so there is no way to place a cell it never saw when the
    kernel/archetypes were built. That is structurally impossible for this node-holdout
    test, not merely unimplemented -- hence no SEACells row anywhere in this notebook.

    stage1z/scvi's own encoders WERE trained on every cell's expression in E1 (full
    dataset, no node split) -- they just never received any affinity-graph supervision.
    That is a real, documented asymmetry against scProto (node-holdout) in this specific
    comparison: these two rows keep the "zero affinity info" framing from
    heldout_edge_modularity.ipynb, but are not being tested on literally-unseen cells the
    way scProto (node-holdout) is. Reported anyway, with the caveat stated in the intro
    markdown cell rather than glossed over here.

    A method missing from disk (E1 not yet run for it on this dataset) prints a warning
    and returns None rather than crashing the whole comparison.
    '''
    tag = f'X_{method}'
    leiden_dir = _find_existing_leiden_dir(ds_id, tag)
    if leiden_dir is None:
        print(f"[{ds_id}] leiden_{tag} not found on disk yet -- skipping this row.")
        return None
    df = pd.read_csv(os.path.join(leiden_dir, 'cell_assignments.csv')).set_index('cell_id')
    s = df['metacell_id'].reindex(cell_index)
    if s.isna().any():
        print(f"[{ds_id}] WARNING: leiden_{tag} assignments missing "
              f"{int(s.isna().sum())} cells after reindex -- skipping this row.")
        return None
    return s.values.astype(int)

## Run one dataset

`run_node_holdout_experiment(ds_id)` does the full pipeline for one dataset: load the
canonical full run (for `ad_full`, the untouched full affinity graph, and `K`), split
cells into train/test nodes, write a train-only AnnData and train scProto on the affinity
graph generated from it alone, encode the held-out test cells through the frozen trained
encoder, then score every method's partition of `ad_full` against the untouched full
graph and against the subset of it touching test nodes (overall, and split into
rare-cell-touching vs. common-only slices).

In [22]:
def run_node_holdout_experiment(ds_id):
    lk = DATASETS[ds_id]['label_key']
    bk = DATASETS[ds_id].get('batch_key')
    kwargs = common_kwargs()

    # --- 1. Load the canonical full run (never retrained here): gives us ad_full, the
    #     untouched full affinity graph this experiment evaluates against, and K to
    #     match so modularity numbers are comparable to the paper's headline table. ---
    t_base, res_base, _ = run_mc_task(ds_id, affinity_type=AFFINITY, load_umap=True, **kwargs)
    ad_full = t_base.train_ds.adata
    aff_full = t_base.train_ds.aff_raw if hasattr(t_base.train_ds, 'aff_raw') else t_base.train_ds.aff
    aff_full = sp.csr_matrix(aff_full)
    n_cells = len(ad_full)
    K = t_base.nmb_prototypes
    print(f"[{ds_id}] {n_cells} cells, K={K} prototypes (matched to the canonical run)")

    # --- 2. Node split, cached to disk so a re-run reloads the exact split a saved
    #     checkpoint was actually trained on, instead of trusting recomputation to be
    #     deterministic across a kernel restart / upstream data changes. ---
    split_path = os.path.join(NODE_HELDOUT_DATA_DIR, f'{ds_id}_{TAG}_split.npz')
    if os.path.exists(split_path):
        d = np.load(split_path, allow_pickle=True)
        train_ids, test_ids = d['train_ids'], d['test_ids']
        train_idx = ad_full.obs_names.get_indexer(train_ids)
        test_idx = ad_full.obs_names.get_indexer(test_ids)
        assert (train_idx >= 0).all() and (test_idx >= 0).all(), (
            f"[{ds_id}] cached split at {split_path} references cells not present in "
            "the current ad_full -- upstream data changed, delete the cache and rerun."
        )
        print(f"[{ds_id}] loaded existing node split from disk (not recomputed): {split_path}")
    else:
        train_idx, test_idx = split_nodes_train_test(
            ad_full, bk, frac_test=FRAC_TEST, min_train_per_batch=MIN_TRAIN_PER_BATCH, seed=SEED,
        )
        np.savez(
            split_path,
            train_ids=ad_full.obs_names.values[train_idx],
            test_ids=ad_full.obs_names.values[test_idx],
        )
        print(f"[{ds_id}] saved node split: {split_path}")

    test_mask = np.zeros(n_cells, dtype=bool)
    test_mask[test_idx] = True

    # --- 3. Train scProto with the affinity graph generated on train nodes ONLY. Test
    #     cells are entirely absent from this AnnData, so they cannot appear as a node
    #     in generate_affinity's kNN/arbf graph, cannot enter the UMAP edge sampler, and
    #     contribute no gradient anywhere at any stage. ---
    train_h5ad_path = os.path.join(NODE_HELDOUT_DATA_DIR, f'{ds_id}_{TAG}_train.h5ad')
    if not os.path.exists(train_h5ad_path):
        ad_full[train_idx].copy().write_h5ad(train_h5ad_path)
        print(f"[{ds_id}] wrote train-only AnnData ({len(train_idx)} cells): {train_h5ad_path}")

    t_nh, res_nh, _ = run_mc_task(
        train_h5ad_path, batch_key=bk, label_key=lk, num_prototypes=K,
        affinity_type=AFFINITY, trainer_kwargs={'experiment_name': f'{TAG}_{ds_id}'},
        load_umap=LOAD_NODEHELDOUT, **kwargs,
    )
    print(f"[{ds_id}] node-holdout run (train-graph modularity, not the headline number): {res_nh}")

    # --- 4. Assignments for every cell in ad_full, train and test alike. Train cells:
    #     the model's own assignment from training. Test cells: a frozen-encoder
    #     forward pass on cells the model has never seen in any form. ---
    assignments_train, _ = t_nh._get_assignments()
    train_assign_s = pd.Series(assignments_train, index=t_nh.train_ds.adata.obs_names)

    ad_test = ad_full[test_idx].copy()
    assignments_test = encode_and_assign_new_cells(t_nh, ad_test)
    test_assign_s = pd.Series(assignments_test, index=ad_test.obs_names)

    combined = pd.concat([train_assign_s, test_assign_s]).reindex(ad_full.obs_names)
    assert not combined.isna().any(), (
        f"[{ds_id}] missing assignment for some cells after combining train+test -- "
        "check obs_names alignment between t_nh.train_ds.adata, ad_test, and ad_full."
    )
    assignments_scproto = combined.values.astype(int)

    # --- 5. Rare-cell mask + test-node-touching subgraph of the untouched full graph ---
    rare_types = get_rare(ad_full, lk)
    rare_mask = ad_full.obs[lk].isin(rare_types).values
    test_edges = subgraph_touching(aff_full, test_mask)
    print(f"[{ds_id}] {int(test_mask.sum())}/{n_cells} test nodes, "
          f"{test_edges.nnz // 2} edges in the full graph touch at least one test node")

    rows = {'scProto (node-holdout)': assignments_scproto}
    for method in TWO_STEP_METHODS:
        disp = METHOD_DISPLAY_NAMES[method]
        leiden_assign = load_two_step_leiden_assignments(ds_id, method, ad_full.obs_names)
        if leiden_assign is not None:
            rows[f'Leiden ({disp})'] = leiden_assign

    # Per-batch modularity, mean +/- std across batches -- the SAME statistic the
    # paper's own headline Table 1 uses (calc_modularity_per_batch: restrict to edges
    # touching a given batch, score Newman modularity on that restricted subgraph, one
    # score per batch). Computed twice: once on the untouched full graph (directly
    # comparable to Table 1's own full-data number) and once further restricted to
    # test-node-touching edges only (the actual generalization number this experiment
    # exists to produce). THESE are the numbers to cite.
    #
    # The *_pooled columns below are a different, NOT-comparable statistic: Newman
    # modularity computed once over the WHOLE graph (every batch together), which lets
    # any method get "free" credit for the graph's own batch structure rather than
    # genuine within-batch community structure -- see the markdown cell after the
    # summary table for the full explanation and a worked example. Kept only for
    # transparency/debugging, explicitly suffixed so they can't be cited by accident.
    records = []
    for name, assign in rows.items():
        assign = np.asarray(assign)
        overall = held_out_same_cluster_rate(test_edges, assign)
        rare = held_out_same_cluster_rate(test_edges, assign, cell_mask=rare_mask, require='any')
        common = held_out_same_cluster_rate(test_edges, assign, cell_mask=~rare_mask, require='all')

        full_mod_mean = full_mod_std = None
        test_mod_mean = test_mod_std = None
        if bk is not None and bk in ad_full.obs.columns:
            full_batch_mod_s = calc_modularity_per_batch(aff_full, assign, ad_full.obs[bk].values)
            full_mod_mean = float(full_batch_mod_s.mean())
            full_mod_std = float(full_batch_mod_s.std())

            test_batch_mod_s = calc_modularity_per_batch(test_edges, assign, ad_full.obs[bk].values)
            test_mod_mean = float(test_batch_mod_s.mean())
            test_mod_std = float(test_batch_mod_s.std())

        records.append({
            'dataset': ds_id,
            'method': name,
            # --- cite these against Table 1: per-batch mean +/- std ---
            'full_graph_modularity_mean': full_mod_mean,
            'full_graph_modularity_std': full_mod_std,
            'test_node_edges_modularity_mean': test_mod_mean,
            'test_node_edges_modularity_std': test_mod_std,
            # --- do NOT cite against Table 1: pooled over the whole graph at once ---
            'full_graph_modularity_pooled': modularity_on_edges(aff_full, assign),
            'test_node_edges_modularity_pooled': modularity_on_edges(test_edges, assign),
            # --- supplementary diagnostics, pooled (no per-batch analog computed,
            #     matching heldout_edge_modularity.ipynb's own reporting convention) ---
            'test_node_edges_same_cluster_rate': overall['rate'],
            'test_node_edges_same_cluster_rate_weighted': overall['weighted_rate'],
            'rare_edge_same_cluster_rate': rare['rate'],
            'rare_edge_n': rare['n_edges'],
            'common_edge_same_cluster_rate': common['rate'],
        })

    return pd.DataFrame.from_records(records)

## Run all 3 datasets

In [23]:
all_results = []
for ds_id in RNA_SEQ_DATASETS:
    print(f"\n=== {ds_id} ===")
    all_results.append(run_node_holdout_experiment(ds_id))

df_node_heldout = pd.concat(all_results, ignore_index=True)
df_node_heldout


=== pancreas ===
dataset is None, loading pancreas
loading pancreas data
✅ Already subsetted to HVGs (4000 genes).
proto_umap_ds-panc_NP220_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [9]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=5.446/23.916/92.201, effk_med=62.4, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/pancreas/pretrain/pretrain_ds-pancreas_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 'pancreas', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'tech'}
📊 EdgeDataset: 1155146 edges
   Weight range: [0.0320, 0.9688]
   umap_steps_per_epoch=500 → 512000 edges/epoch

  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

Saved clusters (16382 cells, label='proto') and 0 metrics to /content/drive/MyDrive/models//pancreas/proto_umap_ds-panc_NP220_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31/clusters.npz
[proto] unused protos: 1/220 (0.45%)
[proto] mean cell-type purity: 0.9125  (size-weighted: 0.9799 ± 0.0521)
[proto] mean batch entropy: 0.3993  (size-weighted: 1.0214 ± 0.5843)


  0%|          | 0/16 [00:00<?, ?it/s]

...

UMAP data saved to /content/drive/MyDrive/models//pbmc-immune_arbf_nodeheldout20_seed0_train/arbf_nodeheldout20_seed0_pbmc-immune_ds-pbmc_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
[pbmc-immune] node-holdout run (train-graph modularity, not the headline number): {'seed': 31, 'purity': 0.8908073368786336, 'niche_purity': None, 'batch_entropy': 0.1973401637520182, 'modularity': 0.6753248222541511, 'coverage': 0.9375, 'dge_rbo_avg': 0.06883109124871642, 'dge_kendall_avg': 0.02486444791391618, 'dge_jaccard_avg': 0.1614566830252197, 'scgraph_corr_avg': 0.8914616247889728, 'scgraph_corr_std': 0.05458567885472591, 'ct_niche_rbo_avg': None, 'aff_compactness_per_batch': {'10X': 0.4483561977811943, 'Freytag': 0.0797756533480282, 'Oetjen': 0.4381126214773128, 'Sun': 0.06362575432741358, 'Villani': 0.12785265074518262}, 'aff_compactness_mean': 1.44320902339468}


  0%|          | 0/27 [00:00<?, ?it/s]

  0%|          | 0/7 [00:00<?, ?it/s]

[pbmc-immune] 6700/33506 test nodes, 468118 edges in the full graph touch at least one test node


,dataset,method,full_graph_modularity_mean,full_graph_modularity_std,test_node_edges_modularity_mean,test_node_edges_modularity_std,full_graph_modularity_pooled,test_node_edges_modularity_pooled,test_node_edges_same_cluster_rate,test_node_edges_same_cluster_rate_weighted,rare_edge_same_cluster_rate,rare_edge_n,common_edge_same_cluster_rate
0,pancreas,scProto (node-holdout),0.616732,0.069150,0.599290,0.072089,0.694896,0.676378,0.696751,0.711901,0.410103,1841,0.699300
1,pancreas,Leiden (scPoli (Stage-1)),0.354957,0.060207,0.358251,0.053393,0.385329,0.385216,0.383410,0.399421,0.423683,1841,0.383052
2,pancreas,Leiden (scVI (Gaussian)),0.339671,0.114053,0.340876,0.116083,0.362849,0.362917,0.353479,0.369440,0.530690,1841,0.351903
3,lung,scProto (node-holdout),0.672227,0.028570,0.655364,0.034429,0.730065,0.714767,0.735421,0.747776,0.715683,18402,0.736284
4,lung,Leiden (scPoli (Stage-1)),0.493748,0.085120,0.495467,0.083442,0.529461,0.532268,0.539358,0.555282,0.689436,18402,0.532793
5,lung,Leiden (scVI (Gaussian)),0.501475,0.084866,0.500866,0.083461,0.547490,0.547977,0.558437,0.574547,0.792033,18402,0.548219
6,pbmc-immune,scProto (node-holdout),0.630307,0.053410,0.620258,0.051334,0.673075,0.661490,0.717853,0.726431,0.655544,16034,0.720063
7,pbmc-immune,Leiden (scPoli (Stage-1)),0.312369,0.213837,0.313783,0.214745,0.265400,0.265948,0.263662,0.271002,0.570787,16034,0.252769
8,pbmc-immune,Leiden (scVI (Gaussian)),0.323220,0.209779,0.325651,0.208546,0.272349,0.274065,0.272504,0.280961,0.612324,16034,0.260452


In [24]:
pd.set_option('display.width', 160)


def fmt_mean_std(mean, std):
    if pd.isna(mean):
        return None
    return f"{mean:.3f} \u00b1 {std:.3f}"


display_df = df_node_heldout.copy()
display_df['full_graph_modularity (mean \u00b1 std -- cite this against Table 1)'] = display_df.apply(
    lambda r: fmt_mean_std(r['full_graph_modularity_mean'], r['full_graph_modularity_std']), axis=1,
)
display_df['test_node_edges_modularity (mean \u00b1 std -- the generalization number)'] = display_df.apply(
    lambda r: fmt_mean_std(r['test_node_edges_modularity_mean'], r['test_node_edges_modularity_std']), axis=1,
)

summary = display_df.set_index(['dataset', 'method'])[
    ['full_graph_modularity (mean \u00b1 std -- cite this against Table 1)',
     'test_node_edges_modularity (mean \u00b1 std -- the generalization number)',
     'test_node_edges_same_cluster_rate', 'rare_edge_same_cluster_rate',
     'common_edge_same_cluster_rate', 'rare_edge_n']
].round(3)
summary

full_graph_modularity (mean ± std -- cite this against Table 1)  \
dataset     method                                                                                      
pancreas    scProto (node-holdout)                                         0.617 ± 0.069                
            Leiden (scPoli (Stage-1))                                      0.355 ± 0.060                
            Leiden (scVI (Gaussian))                                       0.340 ± 0.114                
lung        scProto (node-holdout)                                         0.672 ± 0.029                
            Leiden (scPoli (Stage-1))                                      0.494 ± 0.085                
            Leiden (scVI (Gaussian))                                       0.501 ± 0.085                
pbmc-immune scProto (node-holdout)                                         0.630 ± 0.053                
            Leiden (scPoli (Stage-1))                                      0.312 ± 0.214                
            Leiden (scVI (Gaussian))                                       0.323 ± 0.210                

                                      test_node_edges_modularity (mean ± std -- the generalization number)  test_node_edges_same_cluster_rate  \
dataset     method                                                                                                                              
pancreas    scProto (node-holdout)                                         0.599 ± 0.072                                                0.697   
            Leiden (scPoli (Stage-1))                                      0.358 ± 0.053                                                0.383   
            Leiden (scVI (Gaussian))                                       0.341 ± 0.116                                                0.353   
lung        scProto (node-holdout)                                         0.655 ± 0.034                                                0.735   
            Leiden (scPoli (Stage-1))                                      0.495 ± 0.083                                                0.539   
            Leiden (scVI (Gaussian))                                       0.501 ± 0.083                                                0.558   
pbmc-immune scProto (node-holdout)                                         0.620 ± 0.051                                                0.718   
            Leiden (scPoli (Stage-1))                                      0.314 ± 0.215                                                0.264   
            Leiden (scVI (Gaussian))                                       0.326 ± 0.209                                                0.273   

                                       rare_edge_same_cluster_rate  common_edge_same_cluster_rate  rare_edge_n  
dataset     method                                                                                              
pancreas    scProto (node-holdout)                           0.410                          0.699         1841  
            Leiden (scPoli (Stage-1))                        0.424                          0.383         1841  
            Leiden (scVI (Gaussian))                         0.531                          0.352         1841  
lung        scProto (node-holdout)                           0.716                          0.736        18402  
            Leiden (scPoli (Stage-1))                        0.689                          0.533        18402  
            Leiden (scVI (Gaussian))                         0.792                          0.548        18402  
pbmc-immune scProto (node-holdout)                           0.656                          0.720        16034  
            Leiden (scPoli (Stage-1))                        0.571                          0.253        16034  
            Leiden (scVI (Gaussian))                         0.612                          0.260        16034

In [25]:
print("Pooled modularity (single Newman-modularity score over the WHOLE graph at once) "
      "-- NOT comparable to Table 1, shown for transparency only. See the markdown cell "
      "below for why these run higher than the mean\u00b1std numbers above.")
pooled_summary = df_node_heldout.set_index(['dataset', 'method'])[
    ['full_graph_modularity_pooled', 'test_node_edges_modularity_pooled']
].round(3)
pooled_summary

Pooled modularity (single Newman-modularity score over the WHOLE graph at once) -- NOT comparable to Table 1, shown for transparency only. See the markdown cell below for why these run higher than the mean±std numbers above.


full_graph_modularity_pooled  test_node_edges_modularity_pooled
dataset     method                                                                                    
pancreas    scProto (node-holdout)                            0.695                              0.676
            Leiden (scPoli (Stage-1))                         0.385                              0.385
            Leiden (scVI (Gaussian))                          0.363                              0.363
lung        scProto (node-holdout)                            0.730                              0.715
            Leiden (scPoli (Stage-1))                         0.529                              0.532
            Leiden (scVI (Gaussian))                          0.547                              0.548
pbmc-immune scProto (node-holdout)                            0.673                              0.661
            Leiden (scPoli (Stage-1))                         0.265                              0.266
            Leiden (scVI (Gaussian))                          0.272                              0.274

## Reading the summary table -- `*_pooled` columns are NOT the same statistic as the paper's headline Table 1 number

`full_graph_modularity_pooled` and `test_node_edges_modularity_pooled` pool the **entire
graph** (every batch together) into one Newman modularity number (`modularity_on_edges`).
The paper's headline number is instead the **mean/std of modularity computed separately
per batch** (`calc_modularity_per_batch`), specifically to avoid crediting a method for
the graph's batch structure itself rather than genuine within-batch community structure
(same reasoning as the rebuttal text to Reviewer 3 re: "why report modularity per batch,
not a single pooled number"). The pooled columns run visibly higher than the paper's own
full-data headline modularity for every row here, including the Leiden baselines --
that's the metric, not a leak (see `pooled_summary` above for the side-by-side numbers).

**The `full_graph_modularity (mean \u00b1 std ...)` and `test_node_edges_modularity (mean
\u00b1 std ...)` columns in `summary` above are the ones to cite.** They're computed the
same way as Table 1 (per-batch, then mean/std across batches) -- `full_graph_modularity`
is directly comparable to scProto's own full-data headline modularity, and
`test_node_edges_modularity` is the actual generalization number this experiment exists
to produce. On the last run, node-holdout scProto's `test_node_edges_modularity` landed
at or just under its own full-data headline modularity (e.g. pancreas 0.599 vs. headline
0.621) -- the expected signature of a harder, data-starved, genuinely out-of-sample test,
not an inflated one. Full writeup, the correctness trace (confirms no data leakage via
PCA/graph construction), and reviewer-response-ready framing:
`../experiment-results/node_heldout_modularity_results.md` (update that file's table with
this run's numbers once it finishes).

## Not covered here

- **SEACells row** — structurally excluded, not just omitted (see the intro cell and
  `load_two_step_leiden_assignments`'s docstring): no encoder means no way to place a
  cell it never saw when its kernel/archetypes were built.
- **Harmony / BBKNN / ComBat two-step rows** — not included by default (`TWO_STEP_METHODS`
  only lists `stage1z`/`scvi`, matching what this round's reviewer reply promised), but
  addable with one line each via `load_two_step_leiden_assignments`, reusing E1's
  already-saved Leiden assignments exactly as `heldout_edge_modularity.ipynb` does for
  its own bonus rows.
- **Held-out-*batch*** modularity (E3 in `../experiments_overview.md`) — a different,
  complementary independence test (unseen *cohort*, not just unseen cells within a seen
  cohort), only valid on datasets with a natural reference/query split.
- **Standard scIB metrics** (E9) — label-based, unaffected by any of this graph/node
  masking, complementary rather than required here.
- **Retraining scVI/scPoli-Stage1 on train cells only, to remove the asymmetry noted in
  the intro cell** — considered and deliberately not done: those two rows exist purely as
  "zero affinity supervision" floors reused unmodified from E1, not as a second inductive
  method being fairly compared to scProto's own generalization. Retraining them here would
  quietly turn this into a different experiment (three inductive methods head-to-head)
  without changing what row 1 vs. rows 2-3 is actually meant to show.